# Step 4 — Manually Approved DOCX Text Executor (B.6)

This optional notebook performs **only** two types of edits on a **copy** of a verified Library DOCX: (1) Table title / number, (2) existing treatment-group header label. It **does not add or delete rows, change table geometry, insert Overall columns, change footnotes, or compute clinical statistics**.

**Required inputs:** the `step4_adaptation_plan_outputs.zip` created by Colab 06 and a separate **human-edited approved review packet**. A pending packet from 06 does **not** grant any approvals. Reviewers must read the Plan and choose a permitted replacement text for each approved operation.

All output documents remain **unapproved partial text-edit demonstrations**, NOT completed target TFL Shells. Never replace the official source Library files.

## 1 · Clone official development branch and install package

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git"
GIT_REF = "feature/official-library-colab"
PROJECT_ROOT = Path("/content/tfl-official-step4-replay")
if (PROJECT_ROOT / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", GIT_REF])
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "checkout", "-B", GIT_REF, "FETCH_HEAD"])
elif not PROJECT_ROOT.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GIT_REF,
                           REPO_URL, str(PROJECT_ROOT)])
else:
    raise RuntimeError(f"{PROJECT_ROOT} exists but is not a Git checkout")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from tfl_agent import TFLSession
from tfl_agent.tools.library_loader import load_library
from tfl_agent.agent.step3_retriever import retrieve_templates, template_selection_decision
from tfl_agent.tools.io_utils import write_json
from IPython.display import display, FileLink
import pandas as pd
import json, zipfile, shutil, traceback

git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Using Agent + official Library branch:", GIT_REF, "at", git_sha)
print("Python package:", __import__("tfl_agent").__file__)

## 2 · Upload your latest Adaptation Plan ZIP from Notebook 06

In [ ]:
from google.colab import files
from tfl_agent.agent.docx_text_executor import (
    build_pending_review_packet, execute_approved_docx_edits
)

PLAN_ZIP_NAME = "step4_adaptation_plan_outputs.zip"
candidates = [Path.cwd() / PLAN_ZIP_NAME, Path("/content") / PLAN_ZIP_NAME]
PLAN_ZIP = next((p for p in candidates if p.exists()), None)
if PLAN_ZIP is None:
    print("Upload the plan ZIP downloaded from Notebook 06.")
    uploaded = files.upload()
    chosen = PLAN_ZIP_NAME if PLAN_ZIP_NAME in uploaded else next(iter(uploaded))
    PLAN_ZIP = next((p for p in (Path.cwd()/chosen, Path("/content")/chosen) if p.exists()), None)
assert PLAN_ZIP is not None and PLAN_ZIP.suffix == ".zip"
EXTRACT_ROOT = Path("/content/step4_v4_plan_input")
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PLAN_ZIP) as archive:
    for entry in archive.infolist():
        candidate = (EXTRACT_ROOT / entry.filename).resolve()
        if not candidate.is_relative_to(EXTRACT_ROOT.resolve()):
            raise ValueError("Unsafe ZIP member: " + entry.filename)
    archive.extractall(EXTRACT_ROOT)
valid_ids = sorted(
    p.name for p in EXTRACT_ROOT.iterdir()
    if p.is_dir() and list((p / "adaptation_plans").glob("*_adaptation_plan.json"))
)
print("Target TFL IDs with source-backed plans:", valid_ids)

## 3 · Select ONE Table and inspect its pending review packet. Nothing is approved here.

In [ ]:
TARGET_TFL_ID = "14.3.1.1"  # Default: Adverse Events
assert TARGET_TFL_ID in valid_ids, f"Unknown TFL ID: {TARGET_TFL_ID}"
PLAN_PATH = EXTRACT_ROOT / TARGET_TFL_ID / "adaptation_plans" / f"{TARGET_TFL_ID}_adaptation_plan.json"
plan = json.loads(PLAN_PATH.read_text(encoding="utf-8"))
assert plan["schema_version"] == "adaptation-plan-4.0", (
    "Rerun updated Notebook 06 to generate schema v4 Plans."
)
packet_name = f"{TARGET_TFL_ID}_review_packet_PENDING.json"
pending_path = PLAN_PATH.with_name(packet_name)
if pending_path.exists():
    pending = json.loads(pending_path.read_text(encoding="utf-8"))
else:
    pending = build_pending_review_packet(plan)
    pending_path.write_text(json.dumps(pending, indent=2, ensure_ascii=False),
                            encoding="utf-8")
print("Planner gate:", plan["planning_gate"]["status"])
print("Layout:", plan["adaptation_mode"])
print("Pending text edits:", len(pending["operations"]))
display(FileLink(str(PLAN_PATH), result_html_prefix="Review full Plan JSON: "))
display(FileLink(str(pending_path), result_html_prefix="Get pending Review Packet JSON: "))
assert plan["planning_gate"]["status"] != "blocked", (
    "This target is blocked. Its Plan is diagnostic only. Do not approve edits."
)

## 4 · Explicit human review (outside this notebook)

Open the **pending review packet JSON**, inspect the original Word text and target SAP requirement, then create a *new copy* of the packet manually:

- Set `reviewer` to the real name or role of the person approving the change.
- For **only** the operations they authorize, set `decision` to `approved`, and set `replacement_text` to **one exact value** already shown in `allowed_replacement_texts`.
- Leave all other operations with `decision: "pending"`. If uncertain, do not approve.
- Save the manually edited file as `<TFL_ID>_review_packet_APPROVED.json` and upload it in the following cell.

No code in this notebook automatically changes a pending operation to approved. Even after editing, the Word output is **not clinically approved**.

In [ ]:
print("Upload the HUMAN-EDITED approved packet for:", TARGET_TFL_ID)
uploaded = files.upload()
packet_file = next((Path.cwd()/name for name in uploaded if name.endswith(".json")), None)
assert packet_file is not None and packet_file.is_file(), "No approved review JSON provided"
approved_packet = json.loads(packet_file.read_text(encoding="utf-8"))
if not any(x.get("decision") == "approved" for x in approved_packet.get("operations", [])):
    raise ValueError("No explicitly approved edit in packet")
print("Reviewer:", approved_packet.get("reviewer"))
print("Approved text fragments:",
      [x.get("op_id") for x in approved_packet["operations"]
       if x.get("decision") == "approved"])

## 5 · Execute ONLY human-approved text edits on a new DOCX and save the audit report

In [ ]:
SOURCE_DOCX = (
    PROJECT_ROOT / "library" / "TFL_Template_Library" / "table" /
    f"{plan['source_template_id']}.docx"
)
OUTPUT_DIR = Path("/content/step4_approved_text_copies")
OUTPUT_DOCX = OUTPUT_DIR / f"Table_{TARGET_TFL_ID}_TEXT_ONLY_REVIEW_REQUIRED.docx"
assert SOURCE_DOCX.is_file(), SOURCE_DOCX
report = execute_approved_docx_edits(
    plan, approved_packet,
    source_docx=SOURCE_DOCX,
    output_docx=OUTPUT_DOCX,
)
REPORT_FILE = OUTPUT_DIR / f"{TARGET_TFL_ID}_text_executor_report.json"
REPORT_FILE.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({
    "status": report["status"],
    "applied_count": report["applied_count"],
    "source_docx_unchanged": report["source_docx_unchanged"],
    "clinical_approval": report["clinical_approval"],
    "layout_validation": report["layout_validation"],
}, indent=2))
display(FileLink(str(OUTPUT_DOCX), result_html_prefix="Unapproved TEXT-ONLY Word copy: "))
display(FileLink(str(REPORT_FILE), result_html_prefix="Audit report JSON: "))
print("IMPORTANT: Render/visually inspect the copy and obtain independent clinical/layout approval.")

## Scope boundaries

This is a minimal, manually authorized **bookmark-text executor**, not a structure-preserving Table Shell generator. The original Library remains untouched and hash-verified. It refuses stale approval packets, changed source files, unsupported edit types, unsafe bookmark spans, denied/blocked targets, and outputs written into the source folder.

**Not yet supported:** automatic Row/Column surgery, Target statistical methods or footnotes, patient-level calculations, clinical review approval, or rendered-page/layout validation. Running this notebook is optional; the ordinary Adaptation Planner v4 does not invoke it.